In [0]:
#Analyze employee salary data department-wise and load results into data-lake
#EMPLOYEE TABLE
emp_data=[(1,"Ravi","IT",8000),
           (2,"Priya","HR",4000),
           (3,"Kumar","IT",9000),
           (4,"Meena","Finance",6000),
           (5,"Raja","HR",3000)
         ]
emp_columns=["ID","NAME","DEPT","SALARY"]
emp_df=spark.createDataFrame(emp_data,emp_columns) 

#DEPARTMENT TABLE
dept_data=[(1 ,"IT","Chennai"),
           (2,"HR","Mumbai"),
           (3,"Finance","Bangalore")
           ]
dept_columns=["ID","DEPT","LOCATION"]
dept_df=spark.createDataFrame(dept_data,dept_columns)
emp_df.show()
dept_df.show()

#TRANSFORM -->USED JOIN
joined_df=emp_df.join(dept_df,"DEPT","inner")
joined_df.show()

#FILTER :SAL>5000

filtered_df=joined_df.filter(joined_df["SALARY"]>5000)
filtered_df.show()

#GROUPBY -->DEPT WISE TOOTAL SALARY

from pyspark.sql.functions import sum,collect_list
grouped_df=filtered_df.groupby("DEPT").agg(sum("SALARY").alias("TOTAL_SALARY"),collect_list("NAME").alias("EMPLOYEE"))
grouped_df.show()

from pyspark.sql.functions import col
#SORT
sorted_df=grouped_df.sort(col("TOTAL_SALARY").desc())
sorted_df.show()

#LOAD-DELTA TABLE
sorted_df.write.format("delta").mode("overwrite").saveAsTable("emp_salary_delta")
print("Loaded to delta table")  
spark.sql("select * from emp_salary_delta").show()

#TIME TRAVEL
spark.sql("Describe history emp_salary_delta").show()
spark.sql("select * from emp_salary_delta version as of 1").show()

#MERGE
#1.NEW TABLE
new_data=[(6,["Siva"],"IT",12000)]
new_column=["ID","NAME","DEPT","SALARY"]
new_df=spark.createDataFrame(new_data,new_column)
#2.TEMP VIEW
new_df.createOrReplaceTempView("new_emp")

#3.MERGE
spark.sql("""
          merge into emp_delta as target
          using new_emp as source
          on target.ID=source.ID
          when matched then update set *
          when not matched then insert *
          """)

#4.VERIFY
spark.sql("select * from emp_delta").show()

+---+-----+-------+------+
| ID| NAME|   DEPT|SALARY|
+---+-----+-------+------+
|  1| Ravi|     IT|  8000|
|  2|Priya|     HR|  4000|
|  3|Kumar|     IT|  9000|
|  4|Meena|Finance|  6000|
|  5| Raja|     HR|  3000|
+---+-----+-------+------+

+---+-------+---------+
| ID|   DEPT| LOCATION|
+---+-------+---------+
|  1|     IT|  Chennai|
|  2|     HR|   Mumbai|
|  3|Finance|Bangalore|
+---+-------+---------+

+-------+---+-----+------+---+---------+
|   DEPT| ID| NAME|SALARY| ID| LOCATION|
+-------+---+-----+------+---+---------+
|     IT|  1| Ravi|  8000|  1|  Chennai|
|     HR|  2|Priya|  4000|  2|   Mumbai|
|     IT|  3|Kumar|  9000|  1|  Chennai|
|Finance|  4|Meena|  6000|  3|Bangalore|
|     HR|  5| Raja|  3000|  2|   Mumbai|
+-------+---+-----+------+---+---------+

+-------+---+-----+------+---+---------+
|   DEPT| ID| NAME|SALARY| ID| LOCATION|
+-------+---+-----+------+---+---------+
|     IT|  1| Ravi|  8000|  1|  Chennai|
|     IT|  3|Kumar|  9000|  1|  Chennai|
|Finance|  4